In [1]:
# --- Setup: add project root to sys.path so that src is discoverable ---
import sys
from pathlib import Path
import datetime
import os
project_root = Path.cwd().parent  # assuming current working directory is 'notebook'
sys.path.insert(0, str(project_root))

# --- Create a timestamped results folder ---
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
results_folder = project_root / "results" / f"run_{timestamp}"
results_folder.mkdir(parents=True, exist_ok=True)
print(f"Results will be saved in: {results_folder}")

Results will be saved in: c:\Users\chris\OneDrive\Documents\Academic\CS 370\tx-soil-moisture\new-code\results\run_20250210_144030


In [2]:
# --- Imports from our modules ---
from src.config.loaders import load_config, load_station_data
from src.data.engineering import engineer_data
from src.data.scaling import scale_df, inverse_scale
from src.data.windowing import data_to_X_y
from src.visualization.plotting import plot_loss, plot_predictions
from src.models.lstm import build_original_simple_lstm_model, build_optimized_lstm_model
from src.models.cnn import build_cnn_model

# --- Standard Setup ---
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import pandas as pd

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = False
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [3]:
# --- Load configuration ---
cfg = load_config()

model_builders = {
    "Simple LSTM": build_original_simple_lstm_model,
    "Optimized LSTM": build_optimized_lstm_model,
    "CNN": build_cnn_model
}

In [4]:
# --- Load and preprocess data ---
station_data = load_station_data(cfg)
for station in station_data:
    print(f"Engineering features for {station} ...")
    station_data[station] = engineer_data(station_data[station])

# For forecasting, choose Station1 (for example)
df_forecast = station_data["Station1"]
scaled_df, scaler = scale_df(df_forecast)

Loaded data from 6 stations: ['Station1', 'Station2', 'Station3', 'Station4', 'Station5', 'Station6']
Engineering features for Station1 ...
Engineering features for Station2 ...
Engineering features for Station3 ...
Engineering features for Station4 ...
Engineering features for Station5 ...
Engineering features for Station6 ...


In [5]:
# --- Create Sliding Windows ---
TARGET_COLUMN = cfg["model"]["target_column"]  # "SWC_20"
WINDOW_SIZE = cfg["model"]["window_size"]         # 72
OFFSET = cfg["model"].get("offset", 24)             # 24
X, y = data_to_X_y(scaled_df[TARGET_COLUMN], WINDOW_SIZE, OFFSET)
print("Full sliding windows shapes: X =", X.shape, ", y =", y.shape)

Full sliding windows shapes: X = (52464, 72, 1) , y = (52464,)


In [6]:
# Split into training (70%), validation (20%), test (10%)
n = len(X)
X_train, y_train = X[:int(n*0.7)], y[:int(n*0.7)]
X_val, y_val = X[int(n*0.7):int(n*0.9)], y[int(n*0.7):int(n*0.9)]
X_test, y_test = X[int(n*0.9):], y[int(n*0.9):]
print("Split shapes -- X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)


Split shapes -- X_train: (36724, 72, 1) X_val: (10493, 72, 1) X_test: (5247, 72, 1)


In [7]:
# --- Hyperparameters ---
EPOCHS = cfg["lstm"]["epochs"]
BATCH_SIZE = cfg["lstm"]["batch_size"]

# --- Loop over Models ---
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

results = []  # To store evaluation metrics for each model

for model_name, builder in model_builders.items():
    print(f"\nTraining model: {model_name}")
    
    # Create a unique checkpoint path for the model within the results folder.
    checkpoint_path = results_folder / f"{model_name.replace(' ', '_')}_checkpoint.keras"
    
    # Build the model using the provided builder and window size.
    model = builder(WINDOW_SIZE)
    
    # Setup callbacks for checkpointing and early stopping.
    cp = tf.keras.callbacks.ModelCheckpoint(
        str(checkpoint_path), save_best_only=True, monitor='val_loss', mode='min'
    )
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min')
    
    # Train the model.
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=[cp, early_stopping],
        verbose=1
    )
    
    # Optionally, plot and save the training & validation loss.
    loss_plot_path = results_folder / f"{model_name.replace(' ', '_')}_loss_plot.png"
    plot_loss(history, save_path=str(loss_plot_path))
    
    # Reload the best saved model.
    model = tf.keras.models.load_model(str(checkpoint_path))
    
    # Generate predictions on the test set.
    predictions_scaled = model.predict(X_test).flatten()
    preds_inv_df = inverse_scale(predictions_scaled, scaled_df, TARGET_COLUMN)
    predictions_rescaled = preds_inv_df[TARGET_COLUMN].values
    actual_inv_df = inverse_scale(y_test.reshape(-1, 1), scaled_df, TARGET_COLUMN)
    y_actual = actual_inv_df[TARGET_COLUMN].values
    
    # Compute evaluation metrics.
    r2 = r2_score(y_actual, predictions_rescaled)
    mse = mean_squared_error(y_actual, predictions_rescaled)
    mae = mean_absolute_error(y_actual, predictions_rescaled)
    mape = mean_absolute_percentage_error(y_actual, predictions_rescaled)
    
    results.append({
        "Model": model_name,
        "R2": r2,
        "MSE": mse,
        "MAE": mae,
        "MAPE": mape
    })
    
    # Plot and save predictions vs. actual values.
    predictions_plot_path = results_folder / f"{model_name.replace(' ', '_')}_predictions.png"
    plot_predictions(y_actual, predictions_rescaled, TARGET_COLUMN, save_path=str(predictions_plot_path))
    
    print(f"Finished evaluation for {model_name}:")
    print(f"  R2: {r2:.4f}")
    print(f"  MSE: {mse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  MAPE: {mape:.4f}")


Training model: Simple LSTM
Epoch 1/10


c:\Users\chris\miniconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


1148/1148 ━━━━━━━━━━━━━━━━━━━━ 24s 20ms/step - loss: 0.0316 - root_mean_squared_error: 0.1621 - val_loss: 0.0034 - val_root_mean_squared_error: 0.0579
Epoch 2/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - loss: 0.0041 - root_mean_squared_error: 0.0642 - val_loss: 0.0033 - val_root_mean_squared_error: 0.0572
Epoch 3/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - loss: 0.0041 - root_mean_squared_error: 0.0643 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0565
Epoch 4/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - loss: 0.0040 - root_mean_squared_error: 0.0634 - val_loss: 0.0031 - val_root_mean_squared_error: 0.0561
Epoch 5/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 21s 18ms/step - loss: 0.0038 - root_mean_squared_error: 0.0616 - val_loss: 0.0031 - val_root_mean_squared_error: 0.0557
Epoch 6/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 21s 19ms/step - loss: 0.0039 - root_mean_squared_error: 0.0624 - val_loss: 0.0030 - val_root_mean_squared_error: 0.0552
Epoch 7/10
1148/1148 ━━━━━━━━━━━━━━━━━━

c:\Users\chris\miniconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


1148/1148 ━━━━━━━━━━━━━━━━━━━━ 72s 60ms/step - loss: 0.0226 - root_mean_squared_error: 0.1361 - val_loss: 0.0034 - val_root_mean_squared_error: 0.0580
Epoch 2/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 73s 64ms/step - loss: 0.0042 - root_mean_squared_error: 0.0648 - val_loss: 0.0032 - val_root_mean_squared_error: 0.0567
Epoch 3/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 75s 58ms/step - loss: 0.0039 - root_mean_squared_error: 0.0622 - val_loss: 0.0031 - val_root_mean_squared_error: 0.0555
Epoch 4/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 78s 68ms/step - loss: 0.0040 - root_mean_squared_error: 0.0628 - val_loss: 0.0030 - val_root_mean_squared_error: 0.0551
Epoch 5/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 75s 65ms/step - loss: 0.0037 - root_mean_squared_error: 0.0608 - val_loss: 0.0030 - val_root_mean_squared_error: 0.0550
Epoch 6/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 70s 61ms/step - loss: 0.0037 - root_mean_squared_error: 0.0608 - val_loss: 0.0031 - val_root_mean_squared_error: 0.0561
Epoch 7/10
1148/1148 ━━━━━━━━━━━━━━━━━━

c:\Users\chris\miniconda3\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


1148/1148 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.0817 - root_mean_squared_error: 0.2691 - val_loss: 0.0060 - val_root_mean_squared_error: 0.0774
Epoch 2/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 0.0072 - root_mean_squared_error: 0.0847 - val_loss: 0.0052 - val_root_mean_squared_error: 0.0724
Epoch 3/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0069 - root_mean_squared_error: 0.0831 - val_loss: 0.0052 - val_root_mean_squared_error: 0.0719
Epoch 4/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.0070 - root_mean_squared_error: 0.0836 - val_loss: 0.0051 - val_root_mean_squared_error: 0.0713
Epoch 5/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 0.0067 - root_mean_squared_error: 0.0817 - val_loss: 0.0050 - val_root_mean_squared_error: 0.0709
Epoch 6/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.0067 - root_mean_squared_error: 0.0817 - val_loss: 0.0049 - val_root_mean_squared_error: 0.0701
Epoch 7/10
1148/1148 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/st

In [8]:
# --- Compile and Save Evaluation Metrics ---
results_df = pd.DataFrame(results)
print("\nModel Performance Comparison:")
print(results_df)
results_csv_path = results_folder / "model_comparison_results.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"Model performance results saved to {results_csv_path}")


Model Performance Comparison:
            Model        R2       MSE       MAE      MAPE
0     Simple LSTM  0.912307  0.001696  0.015362  0.041439
1  Optimized LSTM  0.904814  0.001841  0.021651  0.064837
2             CNN  0.877048  0.002377  0.028478  0.083469
Model performance results saved to c:\Users\chris\OneDrive\Documents\Academic\CS 370\tx-soil-moisture\new-code\results\run_20250210_144030\model_comparison_results.csv
